In [ ]:
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, year, to_date, count, when
from pyspark.sql.types import IntegerType, DoubleType

spark = SparkSession.builder \
    .appName("TratamentoYouTube") \
    .getOrCreate()

print("Spark iniciado com sucesso!")

Spark iniciado com sucesso!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving USvideos.csv to USvideos.csv
Saving videos-stats.csv to videos-stats.csv
Saving comments.csv to comments.csv


In [ ]:
import zipfile
import os

zip_name = "Material de apoio - M27.zip"

if os.path.exists(zip_name):
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall("/content/")
    print("Arquivos extraídos com sucesso!")
else:
    print("ZIP não encontrado. Se você enviou os CSVs separadamente, pode seguir normalmente.")

ZIP não encontrado. Se você enviou os CSVs separadamente, pode seguir normalmente.


In [ ]:
videos_file = "/content/videos-stats.csv"
comments_file = "/content/comments.csv"
usvideos_file = "/content/USvideos.csv"

In [ ]:
df_video = spark.read.csv(videos_file, header=True, inferSchema=True)

print("Schema df_video:")
df_video.printSchema()

df_video.show(5, truncate=False)

Schema df_video:
root
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: double (nullable = true)
 |-- Comments: double (nullable = true)
 |-- Views: double (nullable = true)

+---+--------------------------------------------------------------------------------------------------+-----------+------------+-------+-------+--------+---------+
|_c0|Title                                                                                             |Video ID   |Published At|Keyword|Likes  |Comments|Views    |
+---+--------------------------------------------------------------------------------------------------+-----------+------------+-------+-------+--------+---------+
|0  |Apple Pay Is Killing the Physical Wallet After Only Eight Years | Tech News Briefing Podcast | WSJ|wAZZ-UWGVHI|2022-08-23  |tech   |3407.0 |672.0   |135612.0 |
|1  |The 

In [ ]:
df_video = df_video.fillna({
    "Likes": 0,
    "Comments": 0,
    "Views": 0
})

df_video.select("Likes", "Comments", "Views").show(5)

+-------+--------+---------+
|  Likes|Comments|    Views|
+-------+--------+---------+
| 3407.0|   672.0| 135612.0|
|76779.0|  4306.0|1758063.0|
|63825.0|  3338.0|1564007.0|
|71566.0|  1426.0| 922918.0|
|96513.0|  5155.0|1855644.0|
+-------+--------+---------+
only showing top 5 rows


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Define explicit schema for df_comentario
# This schema explicitly sets 'Likes' and 'Sentiment' as StringType
# to prevent inferSchema from misinterpreting '95.0' etc. as doubles directly.
explicit_comments_schema_initial = StructType([
    StructField("_c0", StringType(), True),
    StructField("Video ID", StringType(), True),
    StructField("Comment", StringType(), True),
    StructField("Likes", StringType(), True),      # Read as String to handle '95.0' correctly
    StructField("Sentiment", StringType(), True)  # Read as String to handle '1.0' correctly
])

df_comentario = spark.read.csv(comments_file, header=True, schema=explicit_comments_schema_initial)

print("Schema df_comentario:")
df_comentario.printSchema()

df_comentario.show(5, truncate=False)


Schema df_comentario:
root
 |-- _c0: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Likes: string (nullable = true)
 |-- Sentiment: string (nullable = true)

+---+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+---------+
|_c0|Video ID   |Comment                                                                                                                                                                                                                                                                      

In [ ]:
qtd_video_antes = df_video.count()
qtd_comentario_antes = df_comentario.count()

print("Quantidade de registros em df_video:", qtd_video_antes)
print("Quantidade de registros em df_comentario:", qtd_comentario_antes)

Quantidade de registros em df_video: 1881
Quantidade de registros em df_comentario: 30036


In [ ]:
df_video = df_video.filter(col("Video ID").isNotNull())
df_comentario = df_comentario.filter(col("Video ID").isNotNull())

qtd_video_sem_nulos = df_video.count()
qtd_comentario_sem_nulos = df_comentario.count()

print("Quantidade de registros em df_video após remover Video ID nulo:", qtd_video_sem_nulos)
print("Quantidade de registros em df_comentario após remover Video ID nulo:", qtd_comentario_sem_nulos)

Quantidade de registros em df_video após remover Video ID nulo: 1881
Quantidade de registros em df_comentario após remover Video ID nulo: 22555


In [ ]:
df_video = df_video.dropDuplicates(["Video ID"])

print("Quantidade de registros em df_video após remover duplicados de Video ID:", df_video.count())

Quantidade de registros em df_video após remover duplicados de Video ID: 1869


In [ ]:
from pyspark.sql.types import LongType

df_video = df_video.withColumn("Likes", col("Likes").cast(LongType())) \
                   .withColumn("Comments", col("Comments").cast(LongType())) \
                   .withColumn("Views", col("Views").cast(LongType()))

df_video.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: long (nullable = true)
 |-- Comments: long (nullable = true)
 |-- Views: long (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: integer (nullable = true)



In [ ]:
from pyspark.sql.functions import col, expr
from pyspark.sql.types import IntegerType, DoubleType, LongType

df_comentario = df_comentario.withColumnRenamed("Likes", "Likes Comment")

# Robustly cast 'Likes Comment': try_cast to Double, then cast to LongType
df_comentario = df_comentario.withColumn("Likes Comment", expr("try_cast(`Likes Comment` AS DOUBLE)").cast(LongType()))

# Robustly cast 'Sentiment': try_cast to Double, then cast to Int
df_comentario = df_comentario.withColumn("Sentiment", expr("try_cast(`Sentiment` AS DOUBLE)").cast(IntegerType()))

df_comentario.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Likes Comment: long (nullable = true)
 |-- Sentiment: integer (nullable = true)



In [ ]:
from pyspark.sql.types import LongType

df_video = df_video.withColumn(
    "Interaction",
    (col("Likes") + col("Comments") + col("Views")).cast(LongType())
)

df_video.select("Title", "Likes", "Comments", "Views", "Interaction").show(5, truncate=False)

+-----------------------------------------------------------------------------------------------------+------+--------+--------+-----------+
|Title                                                                                                |Likes |Comments|Views   |Interaction|
+-----------------------------------------------------------------------------------------------------+------+--------+--------+-----------+
|ASMR MUKBANG DOUBLE BIG MAC &amp; CHEESY HASH BROWNS &amp; CHICKEN NUGGETS (No Talking) EATING SOUNDS|378858|18860   |17975269|18372987   |
|Deadly car bomb detonates outside Moscow                                                             |6379  |4853    |808787  |820019     |
|How Biden&#39;s student loan forgiveness program will work                                           |1029  |2347    |97434   |100810     |
|Celebrating My 400 Pound Milestone.... McDonald&#39;s Mukbang                                        |45628 |17264   |5283664 |5346556    |
|Physics Revi

In [ ]:
df_comentario.show(5, truncate=False)

+---+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+---------+
|_c0|Video ID   |Comment                                                                                                                                                                                                                                                                                                                                                                                                                                                                    |Likes|Sentiment|
+---+-------

In [ ]:
df_comentario_verified.dropna().show(5, truncate=False)

+---+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+---------+
|_c0|Video ID   |Comment                                                                                                                                                                                                                                                                                                                                                                                                                                                                    |Likes Comment|Sentime

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import LongType

temp_output_path = "/content/df_comentario_temp_parquet"
df_comentario.write.mode("overwrite").parquet(temp_output_path)

df_comentario_verified = spark.read.parquet(temp_output_path)

# Explicitly cast 'Likes Comment' to LongType after reading from parquet
df_comentario_verified = df_comentario_verified.withColumn("Likes Comment", col("Likes Comment").cast(LongType()))

# Drop the _c0 column from df_comentario_verified as it's a redundant index
if "_c0" in df_comentario_verified.columns:
    df_comentario_verified = df_comentario_verified.drop("_c0")

df_comentario_verified.printSchema()
df_comentario_verified.show(5, truncate=False)

root
 |-- Video ID: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Likes Comment: long (nullable = true)
 |-- Sentiment: integer (nullable = true)

+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+---------+
|Video ID   |Comment                                                                                                                                                                                                                                                                                                                        

In [ ]:
df_comentario.limit(5).toPandas()

,_c0,Video ID,Comment,Likes,Sentiment
0,0,wAZZ-UWGVHI,Let's not forget that Apple Pay in 2014 requir...,95.0,1.0
1,1,wAZZ-UWGVHI,Here in NZ 50% of retailers don’t even have co...,19.0,0.0
2,2,wAZZ-UWGVHI,I will forever acknowledge this channel with t...,161.0,2.0
3,3,wAZZ-UWGVHI,Whenever I go to a place that doesn’t take App...,8.0,0.0
4,4,wAZZ-UWGVHI,"Apple Pay is so convenient, secure, and easy t...",34.0,2.0


In [ ]:
df_video = df_video.withColumn("Published At", to_date(col("Published At"), "yyyy-MM-dd"))

df_video.printSchema()
df_video.select("Published At").show(5, truncate=False)

root
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: long (nullable = true)
 |-- Comments: long (nullable = true)
 |-- Views: long (nullable = true)
 |-- Interaction: long (nullable = true)
 |-- Year: integer (nullable = true)

+------------+
|Published At|
+------------+
|2020-08-18  |
|2021-12-03  |
|2022-08-18  |
|2022-08-18  |
|2021-04-22  |
+------------+
only showing top 5 rows


In [ ]:
df_video = df_video.withColumn("Year", year(col("Published At")))

df_video.select("Published At", "Year").show(5, truncate=False)

+------------+----+
|Published At|Year|
+------------+----+
|2020-08-18  |2020|
|2021-12-03  |2021|
|2022-08-18  |2022|
|2022-08-18  |2022|
|2021-04-22  |2021|
+------------+----+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import col

# Recriando o join com cast preventivo para String para evitar bug de overflow interno do Spark
df_join_video_comments = df_video.join(df_comentario_verified, on="Video ID", how="inner") \
    .withColumn("Views", col("Views").cast("string")) \
    .withColumn("Interaction", col("Interaction").cast("string"))

print("Join recriado. Colunas 'Views' e 'Interaction' convertidas para String para estabilidade.")
df_join_video_comments.printSchema()

Join recriado. Colunas 'Views' e 'Interaction' convertidas para String para estabilidade.
root
 |-- Video ID: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: long (nullable = true)
 |-- Comments: long (nullable = true)
 |-- Views: string (nullable = true)
 |-- Interaction: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Likes Comment: long (nullable = true)
 |-- Sentiment: integer (nullable = true)



In [ ]:
from pyspark.sql.functions import count, when, col, isnan

print("Nulos em df_comentario_verified:")
df_nulos_comentario = df_comentario_verified.select([
    count(when(col(c).isNull() | isnan(c), c)).alias(c)
    if dict(df_comentario_verified.dtypes)[c] in ["double", "float", "int", "bigint", "long"]
    else count(when(col(c).isNull(), c)).alias(c)
    for c in df_comentario_verified.columns
])

# Cast all columns to StringType before converting to Pandas to avoid CAST_OVERFLOW
df_nulos_comentario_pd = df_nulos_comentario.select([col(c).cast("string").alias(c) for c in df_nulos_comentario.columns]).toPandas()
display(df_nulos_comentario_pd)

Nulos em df_comentario_verified:


,Video ID,Comment,Likes Comment,Sentiment
0,7481,8194,14342,14456


In [ ]:
import pandas as pd
from pyspark.sql.functions import col, isnan, count, when

print("Counting nulls column-by-column to avoid global overflow errors:")

results = {}
for c in df_join_video_comments.columns:
    try:
        # Determine if numeric for isnan check
        dtype = dict(df_join_video_comments.dtypes)[c]
        is_numeric = dtype in ["double", "float", "int", "bigint", "long"]

        # Perform count and immediately convert to Python scalar to bypass Spark's internal collection issues
        if is_numeric:
            null_count = df_join_video_comments.filter(col(f"`{c}`").isNull() | isnan(f"`{c}`")).count()
        else:
            null_count = df_join_video_comments.filter(col(f"`{c}`").isNull()).count()

        results[c] = str(null_count)
    except Exception as e:
        results[c] = f"Error: {str(e)[:50]}..."

# Display as a Pandas DataFrame for better formatting
pd_results = pd.DataFrame([results])
display(pd_results)

Counting nulls column-by-column to avoid global overflow errors:


,Video ID,Title,Published At,Keyword,Likes,Comments,Views,Interaction,Year,Comment,Likes Comment,Sentiment
0,0,0,0,0,0,0,Error: [CAST_OVERFLOW] The value 4.034122271E9...,Error: [CAST_OVERFLOW] The value 4.034122271E9...,0,1,3338,3135


In [ ]:
import tempfile
import pandas as pd
from pyspark.sql.functions import col, expr

# Colunas para análise
numeric_cols_to_analyze = ["Likes", "Comments", "Views", "Interaction", "Likes Comment", "Sentiment", "Year"]

print("Calculando estatísticas com isolamento total de tipos (String early-cast):")

for col_name in numeric_cols_to_analyze:
    print(f"\n--- Coluna: {col_name} ---")
    try:
        # Criamos um DF temporário onde a coluna já nasce como String
        # Usamos try_cast para garantir que o Spark não tente otimizações aritméticas perigosas
        temp_df = df_join_video_comments.selectExpr(f"CAST(`{col_name}` AS STRING) as val_str")

        # Agora calculamos o max/min sobre a string (ordem alfabética pode falhar para números,
        # então primeiro convertemos de volta para DOUBLE apenas dentro da agregação SQL)
        stats_df = temp_df.selectExpr(
            "CAST(MAX(CAST(val_str AS DOUBLE)) AS STRING) as max_val",
            "CAST(MIN(CAST(val_str AS DOUBLE)) AS STRING) as min_val"
        )

        with tempfile.TemporaryDirectory() as tmpdir:
            path = f"{tmpdir}/final_res_{col_name}"
            stats_df.write.mode("overwrite").parquet(path)
            result_pd = spark.read.parquet(path).toPandas()

            print(f"Valor Máximo: {result_pd.loc[0, 'max_val']}")
            print(f"Valor Mínimo: {result_pd.loc[0, 'min_val']}")

    except Exception as e:
        print(f"Erro persistente em {col_name}: {str(e)[:200]}")

Calculando estatísticas com isolamento total de tipos (String early-cast):

--- Coluna: Likes ---
Valor Máximo: 1.6445558E7
Valor Mínimo: -1.0

--- Coluna: Comments ---
Valor Máximo: 732818.0
Valor Mínimo: -1.0

--- Coluna: Views ---
Erro persistente em Views: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003

--- Coluna: Interaction ---
Erro persistente em Interaction: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003

--- Coluna: Likes Comment ---
Valor Máximo: 891372.0
Valor Mínimo: 0.0

--- Coluna: Sentiment ---
Valor Máximo: 19518.0
Valor Mínimo: 0.0

--- Coluna: Year ---
Valor Máximo: 2022.0
Valor Mínimo: 2007.0


In [ ]:
from pyspark.sql.functions import max, col

# Lista de colunas LongType para verificar
long_columns = ["Likes", "Comments", "Views", "Interaction", "Likes Comment"]

print("Valores máximos para colunas LongType em df_join_video_comments:")
for col_name in long_columns:
    print(f"  Verificando máximo para {col_name}:")
    try:
        # Convertendo o resultado da agregação para string para evitar overflow na exibição
        df_join_video_comments.agg(max(col(col_name)).cast("string").alias(f"max_{col_name}")).show(truncate=False)
    except Exception as e:
        print(f"  Erro ao processar {col_name}: {str(e)[:100]}...")

Valores máximos para colunas LongType em df_join_video_comments:
  Verificando máximo para Likes:
+---------+
|max_Likes|
+---------+
|16445558 |
+---------+

  Verificando máximo para Comments:
+------------+
|max_Comments|
+------------+
|732818      |
+------------+

  Verificando máximo para Views:
  Erro ao processar Views: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an over...
  Verificando máximo para Interaction:
  Erro ao processar Interaction: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an over...
  Verificando máximo para Likes Comment:
+-----------------+
|max_Likes Comment|
+-----------------+
|891372           |
+-----------------+



In [ ]:
import pandas as pd
from pyspark.sql.functions import col, isnan, count, when

print("Contagem de nulos por coluna (Teste de estabilidade com LongType):")

results = {}
for c in df_join_video_comments.columns:
    try:
        dtype = dict(df_join_video_comments.dtypes)[c]
        is_numeric = dtype in ["double", "float", "int", "bigint", "long"]

        if is_numeric:
            null_count = df_join_video_comments.filter(col(f"`{c}`").isNull() | isnan(f"`{c}`")).count()
        else:
            null_count = df_join_video_comments.filter(col(f"`{c}`").isNull()).count()

        results[c] = str(null_count)
    except Exception as e:
        results[c] = f"Erro: {str(e)[:50]}..."

pd_stability_results = pd.DataFrame([results])
display(pd_stability_results)

Contagem de nulos por coluna (Teste de estabilidade com LongType):


,Video ID,Title,Published At,Keyword,Likes,Comments,Views,Interaction,Year,Comment,Likes Comment,Sentiment
0,0,0,0,0,0,0,Erro: [CAST_OVERFLOW] The value 4.034122271E9D...,Erro: [CAST_OVERFLOW] The value 4.034122271E9D...,0,1,3338,3135


In [ ]:
# Exibindo o esquema detalhado do DataFrame reconstruído
print("Estrutura atual do df_join_video_comments:")
df_join_video_comments.printSchema()

# Verificação específica dos tipos das colunas numéricas de grande volume
for field in df_join_video_comments.schema.fields:
    if field.name in ['Likes', 'Comments', 'Views', 'Interaction', 'Likes Comment']:
        print(f"Coluna: {field.name} | Tipo: {field.dataType}")

Estrutura atual do df_join_video_comments:
root
 |-- Video ID: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: long (nullable = true)
 |-- Comments: long (nullable = true)
 |-- Views: long (nullable = true)
 |-- Interaction: long (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Likes Comment: long (nullable = true)
 |-- Sentiment: integer (nullable = true)

Coluna: Likes | Tipo: LongType()
Coluna: Comments | Tipo: LongType()
Coluna: Views | Tipo: LongType()
Coluna: Interaction | Tipo: LongType()
Coluna: Likes Comment | Tipo: LongType()


In [ ]:
from pyspark.sql.functions import max, col

# List of LongType columns to check
long_columns = ["Likes", "Comments", "Views", "Interaction", "Likes Comment"]

print("Maximum values for LongType columns in df_join_video_comments:")
for col_name in long_columns:
    print(f"  Checking max for {col_name}:")
    # Explicitly cast the max value to string to avoid display overflow
    df_join_video_comments.agg(max(col(col_name)).cast("string").alias(f"max_{col_name}")).show(truncate=False)

Maximum values for LongType columns in df_join_video_comments:
  Checking max for Likes:
+---------+
|max_Likes|
+---------+
|16445558 |
+---------+

  Checking max for Comments:
+------------+
|max_Comments|
+------------+
|732818      |
+------------+

  Checking max for Views:


ArithmeticException: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003

In [ ]:
from pyspark.sql.functions import expr, col, mean, max, min
import pandas as pd

# Lista de colunas para validação numérica profunda
numeric_cols = ['Likes', 'Comments', 'Views', 'Interaction', 'Likes Comment', 'Sentiment']

print('Gerando Relatório Estatístico Seguro (Bypassing Overflow com try_cast):')

stats_results = []

for c in numeric_cols:
    try:
        # Executamos a agregação usando try_cast para DOUBLE para evitar overflow de INT interno
        # Convertemos o resultado para String logo após o cálculo para garantir a coleta (collect) estável
        query = f"""
            SELECT
                '{c}' as Coluna,
                CAST(AVG(try_cast(`{c}` AS DOUBLE)) AS STRING) as Media,
                CAST(MAX(try_cast(`{c}` AS DOUBLE)) AS STRING) as Maximo,
                CAST(MIN(try_cast(`{c}` AS DOUBLE)) AS STRING) as Minimo
            FROM __table__
        """

        # Criamos uma view temporária para facilitar a query SQL
        df_join_video_comments.createOrReplaceTempView("__table__")
        row = spark.sql(query).collect()[0]

        stats_results.append({
            "Coluna": row["Coluna"],
            "Média": row["Media"],
            "Máximo": row["Maximo"],
            "Mínimo": row["Minimo"]
        })
    except Exception as e:
        stats_results.append({"Coluna": c, "Erro": str(e)[:100]})

# Exibição final com Pandas para melhor formatação
display(pd.DataFrame(stats_results))

Gerando Relatório Estatístico Seguro (Bypassing Overflow com try_cast):


,Coluna,Média,Máximo,Mínimo,Erro
0,Likes,173349.5687978706,1.6445558E7,-1.0,NaN
1,Comments,8022.0688793524905,732818.0,-1.0,NaN
2,Views,NaN,NaN,NaN,[CAST_OVERFLOW] The value 4.034122271E9D of th...
3,Interaction,NaN,NaN,NaN,[CAST_OVERFLOW] The value 4.034122271E9D of th...
4,Likes Comment,1068.1289894499369,891372.0,0.0,NaN
5,Sentiment,14.42182794290952,19518.0,0.0,NaN


In [ ]:
from pyspark.sql.functions import col, length
import pandas as pd

print("Reconstruindo join com isolamento total de tipos (String isolation):")

# Convertemos os DataFrames base para string em todas as colunas numéricas de risco
df_video_text = df_video.select([col(c).cast("string") if c in ['Views', 'Interaction'] else col(c) for c in df_video.columns])
df_comentario_text = df_comentario_verified # Já está estável

# Realizamos o join sobre os dados 'textualizados'
df_join_safe = df_video_text.join(df_comentario_text, on="Video ID", how="inner")

# Agora o count e o ranking devem funcionar sem disparar o erro aritmético
try:
    print("\n--- Relatório Final de Qualidade ---")
    null_counts = {}
    # Verificamos apenas colunas essenciais para evitar sobrecarga
    columns_to_check = ['Title', 'Views', 'Sentiment']
    for c in columns_to_check:
        null_counts[c] = df_join_safe.filter(col(c).isNull()).count()

    display(pd.DataFrame([null_counts]))

    print("\n--- Top 5 Vídeos (Ranking Seguro) ---")
    # Ordenação por length garante que números maiores fiquem no topo
    df_join_safe.select("Title", "Views") \
        .orderBy(length("Views").desc(), col("Views").desc()) \
        .show(5, truncate=False)
except Exception as e:
    print(f"Erro na execução final: {str(e)}")

Reconstruindo join com isolamento total de tipos (String isolation):

--- Relatório Final de Qualidade ---
Erro na execução final: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003


In [ ]:
from pyspark.sql.functions import col, length
import pandas as pd

# 1. Recriando o join com uma estratégia de 'quebra de linhagem'
# Usamos Decimal(20,0) que suporta os 4 bilhões+ sem arredondamento ou overflow de INT
temp_path_safe = "/content/df_join_safe_final"

df_join_final = df_video.join(df_comentario_verified, on="Video ID", how="inner") \
    .withColumn("Views", col("Views").cast("decimal(20,0)").cast("string")) \
    .withColumn("Interaction", col("Interaction").cast("decimal(20,0)").cast("string"))

# Salvamos fisicamente para forçar o Spark a esquecer o plano de execução problemático
df_join_final.write.mode("overwrite").parquet(temp_path_safe)

# Recarregamos o DataFrame 'limpo'
df_final_estavel = spark.read.parquet(temp_path_safe)

print("DataFrame estabilizado com sucesso!")

# 2. Relatório de Nulos Seguro
print("\n--- Relatório de Qualidade (Nulos) ---")
null_results = {}
for c in df_final_estavel.columns:
    null_results[c] = str(df_final_estavel.filter(col(f'`{c}`').isNull()).count())
display(pd.DataFrame([null_results]))

# 3. Visualização dos Maiores Valores
print("\n--- Top 5 Vídeos por Visualizações ---")
# Como são strings, ordenamos pelo tamanho do texto primeiro para garantir ordem numérica correta
df_final_estavel.select('Title', 'Views') \
    .orderBy(length(col('Views')).desc(), col('Views').desc()) \
    .show(5, truncate=False)

ArithmeticException: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003

In [ ]:
from pyspark.sql.functions import col, length, expr
import pandas as pd

print('Reconstruindo Join com Estabilidade Numérica (Decimal Path):')

# Aplicamos a conversão robusta na fonte para evitar que o Spark use tipos INT32 internamente
df_video_fixed = df_video.withColumn('Views', col('Views').cast('decimal(20,0)').cast('string')) \
                         .withColumn('Interaction', col('Interaction').cast('decimal(20,0)').cast('string'))

# Novo join baseado nos dados estabilizados
df_join_estavel = df_video_fixed.join(df_comentario_verified, on='Video ID', how='inner')

print('\n--- Relatório Final de Qualidade (Nulos) ---')
null_summary = {}
for c in ['Title', 'Views', 'Interaction', 'Sentiment']:
    null_summary[c] = df_join_estavel.filter(col(c).isNull()).count()
display(pd.DataFrame([null_summary]))

print('\n--- Top 5 Vídeos por Visualizações (Ranking Seguro) ---')
# Ordenamos pelo comprimento da string primeiro para garantir ordem numérica correta
df_join_estavel.select('Title', 'Views') \
    .orderBy(length(col('Views')).desc(), col('Views').desc()) \
    .show(5, truncate=False)

Reconstruindo Join com Estabilidade Numérica (Decimal Path):

--- Relatório Final de Qualidade (Nulos) ---


ArithmeticException: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003

In [ ]:
from pyspark.sql.functions import col, length
import pandas as pd

print('Extraindo Recordistas de Visualizações e Interações (Estratégia de Ordenação):')

# Como as colunas problemáticas no df_join_video_comments são Strings,
# ordenamos pelo tamanho (length) e depois pelo valor alfabético para simular ordem numérica.
for c in ['Views', 'Interaction']:
    print(f'\n--- Top 1 para a coluna: {c} ---')
    try:
        df_join_video_comments.select('Title', c) \
            .orderBy(length(col(c)).desc(), col(c).desc()) \
            .show(1, truncate=False)
    except Exception as e:
        print(f'Erro na ordenação de {c}: {str(e)[:100]}')

print('\nContagem Final de Registros no Join:')
try:
    # Apenas contagem total para confirmar integridade
    print(f'Total de registros: {df_join_video_comments.count()}')
except Exception as e:
    print(f'Erro na contagem: {str(e)[:100]}')

Extraindo Recordistas de Visualizações e Interações (Estratégia de Ordenação):

--- Top 1 para a coluna: Views ---
Erro na ordenação de Views: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an over

--- Top 1 para a coluna: Interaction ---
Erro na ordenação de Interaction: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an over

Contagem Final de Registros no Join:
Total de registros: 18409


In [ ]:
from pyspark.sql.functions import col

print('Verificação dos Maiores Valores (Top 5):')

# Ordenação baseada em String (funciona bem para identificar os maiores números se não houver zeros à esquerda)
for c in ['Views', 'Interaction']:
    print(f'\n--- Top 5 para a coluna: {c} ---')
    df_join_video_comments.select('Title', c).orderBy(col(c).desc()).show(5, truncate=False)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


Verificação dos Maiores Valores (Top 5):

--- Top 5 para a coluna: Views ---


KeyboardInterrupt: 

In [ ]:
from pyspark.sql.functions import expr, col, length
import pandas as pd

# Usamos try_cast diretamente na expressão SQL para garantir que o Spark
# não tente conversões implícitas para INT em nenhum estágio do plano de execução.
df_final_robusto = df_video.join(df_comentario_verified, on="Video ID", how="inner") \
    .withColumn("Views_Safe", expr("try_cast(Views AS DECIMAL(20,0))")) \
    .withColumn("Interaction_Safe", expr("try_cast(Interaction AS DECIMAL(20,0))"))

# Selecionamos e convertemos para String para garantir exibição total
df_display = df_final_robusto.select(
    "*",
    col("Views_Safe").cast("string").alias("Views_Final"),
    col("Interaction_Safe").cast("string").alias("Interaction_Final")
).drop("Views", "Interaction", "Views_Safe", "Interaction_Safe")

print("Análise de Nulos (Bypassing Overflow):")
# Contamos apenas 1000 registros para garantir resposta rápida e estável
res_nulos = {}
for c in ["Views_Final", "Interaction_Final", "Sentiment"]:
    res_nulos[c] = df_display.filter(col(c).isNull()).count()

print(pd.Series(res_nulos))

print("\nTop 5 Vídeos (Ordenação Segura por String/Length):")
df_display.select("Title", "Views_Final") \
    .orderBy(length("Views_Final").desc(), col("Views_Final").desc()) \
    .show(5, truncate=False)

Análise de Nulos (Bypassing Overflow):


ArithmeticException: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003

In [ ]:
import pandas as pd
from pyspark.sql.functions import col

print('Relatório Final de Qualidade (Contagem de Nulos):')

# Como as colunas problemáticas agora são Strings, o count() deve rodar sem tentar cast para INT
final_nulls = {}
for column in df_join_video_comments.columns:
    try:
        count_val = df_join_video_comments.filter(col(f'`{column}`').isNull()).count()
        final_nulls[column] = str(count_val)
    except Exception as e:
        final_nulls[column] = f"Erro: {str(e)[:50]}"

display(pd.DataFrame([final_nulls]))

Relatório Final de Qualidade (Contagem de Nulos):


,Video ID,Title,Published At,Keyword,Likes,Comments,Views,Interaction,Year,Comment,Likes Comment,Sentiment
0,0,0,0,0,0,0,Erro: [CAST_OVERFLOW] The value 4.034122271E9D...,Erro: [CAST_OVERFLOW] The value 4.034122271E9D...,0,1,3338,3135


In [ ]:
from pyspark.sql.functions import col
import pandas as pd

summary_df = df_join_video_comments.summary()

# Get all column names from the summary_df
all_summary_cols = summary_df.columns

# Cast ALL columns in summary_df to StringType before attempting to write/collect
# This is a precaution, though the bug seems to be deeper.
for c in all_summary_cols:
    summary_df = summary_df.withColumn(c, col(c).cast("string"))

# Try writing to Parquet instead of CSV, then read with Pandas to bypass Spark's internal collect() issue
temp_parquet_path = "/content/summary_temp_parquet"
summary_df.write.mode("overwrite").parquet(temp_parquet_path)

# Read the Parquet back into a Pandas DataFrame
pd_summary_df = spark.read.parquet(temp_parquet_path).toPandas()

pd_summary_df

ArithmeticException: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003

In [ ]:
import pandas as pd

print("Executando Isolamento de Linhagem via RDD para Gráfico...")

try:
    # Extraímos apenas os dados necessários como uma lista de tuplas simples (Python nativo)
    # O limit(10) garante que a coleta seja minúscula e rápida
    raw_data = df_display.select("Title", "Views_Final").limit(10).rdd.map(lambda x: (x[0], x[1])).collect()

    # Criamos o DataFrame Pandas diretamente da lista Python
    pd_plot_data = pd.DataFrame(raw_data, columns=["Title", "Views_Final"])

    # Conversão segura no ambiente Pandas
    pd_plot_data['Views_Numeric'] = pd.to_numeric(pd_plot_data['Views_Final'], errors='coerce')

    print("Dados isolados com sucesso!")
    display(pd_plot_data.head())
except Exception as e:
    print(f"Erro no isolamento: {str(e)}")

Executando Isolamento de Linhagem via RDD para Gráfico...
Erro no isolamento: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'pd_plot_data' in locals() and not pd_plot_data.empty:
    plt.figure(figsize=(12, 6))
    sns.barplot(data=pd_plot_data, x='Views_Numeric', y='Title', palette='magma')
    plt.title('Top 10 Vídeos por Visualizações (Isolamento de RDD)')
    plt.xlabel('Visualizações')
    plt.ylabel('Título do Vídeo')
    plt.tight_layout()
    plt.show()
else:
    print("Erro: Dados não disponíveis para plotagem.")

Erro: Dados não disponíveis para plotagem.


In [ ]:
df_us_videos = spark.read.csv(usvideos_file, header=True, inferSchema=True)

print("Schema df_us_videos:")
df_us_videos.printSchema()

df_us_videos.show(5, truncate=False)

Schema df_us_videos:
root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)

+-----------+-------------+--------------------------------------------------------------+---------------------+-----------+------------------------+---------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
df_us_videos = df_us_videos.withColumnRenamed("title", "Title")

In [ ]:
from pyspark.sql.functions import col, expr

# 1. Aplicamos a conversao robusta na fonte (Decimal 20,0 suporta ate 10^20)
df_video_fixed = df_video.withColumn("Views", expr("try_cast(Views AS DECIMAL(20,0))")) \
                         .withColumn("Interaction", expr("try_cast(Interaction AS DECIMAL(20,0))"))

df_us_videos_renamed = df_us_videos.withColumnRenamed("views", "us_views") \
                                   .withColumnRenamed("likes", "us_likes") \
                                   .withColumnRenamed("comment_count", "us_comments")

# Garantimos que as colunas de views no us_videos tambem usem Decimal
df_us_videos_renamed = df_us_videos_renamed.withColumn("us_views", expr("try_cast(us_views AS DECIMAL(20,0))"))

# 2. Realizamos o Join com os dados ja protegidos contra overflow de INT
df_join_video_usvideos = df_video_fixed.join(df_us_videos_renamed, on="Title", how="inner")

print("Join realizado com protecao de overflow na fonte!")

Join realizado com protecao de overflow na fonte!


In [ ]:
import pandas as pd

print("Visualizacao via RDD (Ignorando limitacoes de CAST do Spark SQL):")

# Extraimos os dados como tuplas Python nativas para evitar o motor de CAST do Spark SQL
try:
    # Selecionamos apenas as colunas desejadas e convertemos para RDD
    # Convertemos cada linha para dicionario para facilitar a criacao do DataFrame Pandas
    raw_data = df_join_video_usvideos.select(
        "Title",
        "Views",
        "us_views",
        "Interaction"
    ).limit(5).rdd.map(lambda row: row.asDict()).collect()

    # Criamos um DataFrame Pandas para exibicao limpa
    df_final_pd = pd.DataFrame(raw_data)

    # Exibicao no ambiente Colab
    display(df_final_pd)

except Exception as e:
    print(f"Erro no isolamento via RDD: {str(e)[:200]}")

Visualizacao via RDD (Ignorando limitacoes de CAST do Spark SQL):
Erro no isolamento via RDD: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003


In [ ]:
import pandas as pd
from pyspark.sql.functions import col

print("Contagem de nulos (Estratégia de Isolamento por Filtro):")

# Lista de colunas para verificar
columns_to_check = df_video.columns
null_results = {}

for c in columns_to_check:
    try:
        # Contamos os nulos filtrando individualmente cada coluna.
        # Isso isola o erro de overflow e permite que ele seja ignorado nas colunas problemáticas.
        count_val = df_video.filter(col(f'`{c}`').isNull()).count()
        null_results[c] = count_val
    except Exception as e:
        # Se houver overflow (como em Views), tentamos converter para string apenas essa coluna
        try:
            count_val = df_video.select(col(f'`{c}`').cast("string")).filter(col(f'`{c}`').isNull()).count()
            null_results[c] = count_val
        except:
            null_results[c] = "Erro de Processamento"

# Exibição final em formato amigável
display(pd.DataFrame([null_results]))

Contagem de nulos (Estratégia de Isolamento por Filtro):


,Title,Video ID,Published At,Keyword,Likes,Comments,Views,Interaction,Year
0,0,0,0,0,0,0,Erro de Processamento,Erro de Processamento,0


In [ ]:
import pandas as pd

print("Extração de Dados via SQL Direto (Isolamento de Plano de Execução):")

try:
    # Registramos o DataFrame como uma tabela temporária para usar SQL puro
    df_join_video_usvideos.createOrReplaceTempView("v_trending")

    # Usamos o SQL para garantir que a conversão ocorra sem interferência do otimizador de DataFrames
    sql_query = """
    SELECT
        Title,
        CAST(CAST(Views AS DECIMAL(20,0)) AS STRING) as Views_Str,
        CAST(CAST(us_views AS DECIMAL(20,0)) AS STRING) as Trending_Views_Str
    FROM v_trending
    LIMIT 10
    """

    # Coletamos os dados como dicionários Python para evitar o driver de exibição do Spark
    raw_rows = spark.sql(sql_query).collect()

    # Criamos um DataFrame Pandas localmente com os dados já extraídos
    pd_final = pd.DataFrame([row.asDict() for row in raw_rows])

    display(pd_final)

except Exception as e:
    print(f"Erro na extração SQL: {str(e)[:250]}")

Extração de Dados via SQL Direto (Isolamento de Plano de Execução):
Erro na extração SQL: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003


In [ ]:
import pandas as pd
from pyspark.sql.functions import expr

print("Visualizacao com Isolamento de String (Bypass de Overflow via SQL):")

try:
    # Aplicamos a conversao robusta em uma nova transformacao para nao afetar as celulas anteriores
    # O try_cast para DECIMAL(20,0) garante que o Spark suporte os 4 bilhoes
    # O CAST final para STRING impede que o driver tente converter para INT32 na coleta
    df_safe_report = df_join_video_usvideos.selectExpr(
        "Title",
        "CAST(try_cast(Views AS DECIMAL(20,0)) AS STRING) as Views_Total",
        "CAST(try_cast(us_views AS DECIMAL(20,0)) AS STRING) as Views_Trending",
        "CAST(try_cast(Interaction AS DECIMAL(20,0)) AS STRING) as Total_Interactions"
    ).limit(10)

    # Convertendo para Pandas para uma exibicao estavel no Colab
    df_final_pd = df_safe_report.toPandas()
    display(df_final_pd)

except Exception as e:
    print(f"Erro ao processar: {str(e)[:250]}")

Visualizacao com Isolamento de String (Bypass de Overflow via SQL):
Erro ao processar: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003


In [ ]:
if "_c0" in df_video.columns:
    df_video = df_video.drop("_c0")

if "Unnamed: 0" in df_video.columns:
    df_video = df_video.drop("Unnamed: 0")

df_video.printSchema()

root
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: long (nullable = true)
 |-- Comments: long (nullable = true)
 |-- Views: long (nullable = true)
 |-- Interaction: long (nullable = true)
 |-- Year: integer (nullable = true)



In [ ]:
from pyspark.sql.functions import col

output_video = "/content/videos-tratados-parquet"

# 1. Preparamos o DataFrame com as colunas problemáticas como String
df_temp = df_video.withColumn("Views", col("Views").cast("string")) \
                  .withColumn("Interaction", col("Interaction").cast("string"))

# 2. QUEBRA DE LINHAGEM: Convertemos para RDD e voltamos para DataFrame
# Isso remove qualquer metadado numérico residual que dispara o overflow interno
df_video_final_save = spark.createDataFrame(df_temp.rdd, schema=df_temp.schema)

# 3. Salvamento agora deve ser imune ao erro aritmético
df_video_final_save.write.mode("overwrite").parquet(output_video)

print("Arquivo parquet de df_video salvo com sucesso via isolamento de RDD em:", output_video)

ArithmeticException: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003

In [ ]:
if "_c0" in df_join_video_comments.columns:
    df_join_video_comments = df_join_video_comments.drop("_c0")

if "Unnamed: 0" in df_join_video_comments.columns:
    df_join_video_comments = df_join_video_comments.drop("Unnamed: 0")

df_join_video_comments.printSchema()

root
 |-- Video ID: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: long (nullable = true)
 |-- Comments: long (nullable = true)
 |-- Views: string (nullable = true)
 |-- Interaction: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Likes Comment: long (nullable = true)
 |-- Sentiment: integer (nullable = true)



In [ ]:
import pandas as pd
from pyspark.sql.functions import col

output_video_comments = "/content/videos-comments-tratados-parquet"

try:
    # 1. Criamos uma versao onde forçamos o Spark a tratar Views e Interaction como String
    # Usamos DECIMAL(20,0) intermediário para suportar os 4 bilhões sem erro de CAST
    df_final_safe = df_join_video_comments.selectExpr(
        "*",
        "CAST(try_cast(Views AS DECIMAL(20,0)) AS STRING) as Views_String",
        "CAST(try_cast(Interaction AS DECIMAL(20,0)) AS STRING) as Interaction_String"
    ).drop("Views", "Interaction")

    # 2. Salvamos em Parquet. Como agora sao Strings, o Spark nao tentará o cast para INT32
    df_final_safe.write.mode("overwrite").parquet(output_video_comments)

    print(f"Sucesso! O arquivo foi salvo em: {output_video_comments}")

    # 3. Agora podemos tentar zipar novamente
    import shutil
    shutil.make_archive(output_video_comments, 'zip', output_video_comments)
    print("Arquivo ZIP gerado com sucesso!")

except Exception as e:
    print(f"Erro ao processar salvamento: {str(e)}")

Erro ao processar salvamento: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003


In [ ]:
print("Quantidade final df_video:", df_video.count())
print("Quantidade final df_comentario:", df_comentario.count())
print("Quantidade final df_join_video_comments:", df_join_video_comments.count())
print("Quantidade final df_join_video_usvideos:", df_join_video_usvideos.count())

Quantidade final df_video: 1869
Quantidade final df_comentario: 30036
Quantidade final df_join_video_comments: 18409
Quantidade final df_join_video_usvideos: 18


In [ ]:
import shutil

shutil.make_archive("/content/videos-tratados-parquet", 'zip', "/content/videos-tratados-parquet")
shutil.make_archive("/content/videos-comments-tratados-parquet", 'zip', "/content/videos-comments-tratados-parquet")

print("Arquivos zipados com sucesso!")

FileNotFoundError: [Errno 2] No such file or directory: '/content/videos-comments-tratados-parquet'

In [ ]:
import pandas as pd

print("Visualização por Extração Nativa (RDD Isolation):")

try:
    # Selecionamos apenas as colunas que queremos ver
    # Convertemos para RDD e usamos uma função lambda para pegar os valores brutos
    # Isso evita que o Spark tente converter tipos durante a serialização do DataFrame
    preview_rdd = df_join_video_comments.select('Video ID', 'Title', 'Views', 'Interaction', 'Sentiment').limit(5).rdd

    # Coletamos os dados como tuplas simples de Python
    data_raw = preview_rdd.map(lambda x: (str(x[0]), str(x[1]), str(x[2]), str(x[3]), str(x[4]))).collect()

    # Criamos o DataFrame Pandas para exibição amigável
    columns = ['Video ID', 'Title', 'Views', 'Interaction', 'Sentiment']
    df_preview_final = pd.DataFrame(data_raw, columns=columns)

    display(df_preview_final)

except Exception as e:
    print(f"Erro na extração bruta: {str(e)[:250]}")

Visualização por Extração Nativa (RDD Isolation):
Erro na extração bruta: [CAST_OVERFLOW] The value 4.034122271E9D of the type "DOUBLE" cannot be cast to "INT" due to an overflow. Use `try_cast` to tolerate overflow and return NULL instead. SQLSTATE: 22003


In [ ]:
import os

print('Listagem de arquivos e pastas em /content:')
for item in os.listdir('/content/'):
    full_path = os.path.join('/content/', item)
    if os.path.isdir(full_path):
        print(f'[DIRETÓRIO] {item}')
    else:
        print(f'[ARQUIVO]   {item}')

Listagem de arquivos e pastas em /content:
[DIRETÓRIO] .config
[DIRETÓRIO] videos-tratados-parquet
[ARQUIVO]   videos-stats.csv
[DIRETÓRIO] df_comentario_temp_parquet
[ARQUIVO]   comments.csv
[ARQUIVO]   videos-tratados-parquet.zip
[ARQUIVO]   USvideos.csv
[DIRETÓRIO] sample_data


In [ ]:
from google.colab import files

files.download("/content/videos-tratados-parquet.zip")
files.download("/content/videos-comments-tratados-parquet.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FileNotFoundError: Cannot find file: /content/videos-comments-tratados-parquet.zip